In [1]:
# ============================================================
# STEP 4 - EXPERIMENT 2
# Cross-Sectional Momentum Benchmark
#
# INPUTS expected in the same working folder:
#   REQUIRED:
#     - experiment_input.zip
#
#   OPTIONAL:
#     - tda_step4_exp1_outputs.zip
#       If present, this code uses Experiment 1's audited ranking
#       to identify the representative-state comparison rows.
#       If absent, it falls back to Step 3 outputs inside experiment_input.zip.
#
# OUTPUT folder:
#   - tda_step4_exp2_outputs/
#
# OUTPUT zip:
#   - tda_step4_exp2_outputs.zip
#
# REQUIRED OUTPUTS:
#   - step4_momentum_benchmark_metrics.csv
#   - step4_momentum_benchmark_daily_nav.csv
#   - step4_momentum_benchmark_daily_returns.csv
#   - step4_momentum_benchmark_rebalance_log.csv
#   - step4_momentum_vs_representative_state_table.md
# ============================================================

import os
import re
import json
import math
import shutil
import zipfile
import hashlib
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

try:
    import yfinance as yf
except ImportError as e:
    raise ImportError("Missing dependency: yfinance. Install with: pip install yfinance") from e


# ============================================================
# 0. CONFIGURATION
# ============================================================

INPUT_ZIP = Path("experiment_input.zip")
STEP4_EXP1_ZIP = Path("tda_step4_exp1_outputs.zip")

OUTPUT_ROOT = Path("tda_step4_exp2_outputs")
WORK_ROOT = Path("_step4_exp2_work")

INITIAL_NAV_DEFAULT = 1_000_000.0
WINDOW_LOOKBACK_DEFAULT = 504
TRANSACTION_COST_RATE_DEFAULT = 0.001

REBALANCE_FREQUENCIES = [21, 42, 63]
SELECTION_FRACTIONS = [0.20, 0.30, 0.40]
MOMENTUM_LOOKBACKS = [21, 63, 126]

REQUIRED_OUTPUTS = [
    "step4_momentum_benchmark_metrics.csv",
    "step4_momentum_benchmark_daily_nav.csv",
    "step4_momentum_benchmark_daily_returns.csv",
    "step4_momentum_benchmark_rebalance_log.csv",
    "step4_momentum_vs_representative_state_table.md",
]

CORE_METHODS = ["Mapper", "Global DBSCAN", "PCA-KMeans"]
BENCHMARK_METHODS = ["Equal Weight Universe", "S&P 500"]

BALANCED_SCORE_WEIGHTS = {
    "Annualized Return Score": 0.35,
    "Daily Sharpe Score": 0.35,
    "Daily Max Drawdown Score": 0.20,
    "Average Turnover Score": 0.10,
}


# ============================================================
# 1. GENERAL HELPERS
# ============================================================

def sha256_file(path: Path) -> str:
    if not path.exists():
        return ""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def clean_folder(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def extract_zip(zip_path: Path, dest: Path):
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(dest)


def extract_all_nested_zips(root: Path):
    nested_zip_paths = list(root.rglob("*.zip"))
    extracted_dirs = []
    for zp in nested_zip_paths:
        target = zp.parent / f"__extracted__{zp.stem}"
        if target.exists():
            shutil.rmtree(target)
        target.mkdir(parents=True, exist_ok=True)
        extract_zip(zp, target)
        extracted_dirs.append(target)
    return extracted_dirs


def find_first_file(root: Path, filename: str):
    matches = list(root.rglob(filename))
    if not matches:
        return None
    return matches[0]


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    unnamed = [c for c in df.columns if c.startswith("Unnamed")]
    if unnamed:
        df = df.drop(columns=unnamed)
    return df


def normalize_symbol(ticker):
    return str(ticker).strip().replace(".", "-")


def dtstr(x):
    if pd.isna(x):
        return ""
    return pd.Timestamp(x).strftime("%Y-%m-%d")


def strategy_name(method, h, l=None):
    if l is None or pd.isna(l):
        return f"{method}, h={int(h)}"
    return f"{method}, h={int(h)}, l={int(round(float(l) * 100))}%"


def momentum_strategy_name(momentum_lookback, h, l):
    return f"Momentum {int(momentum_lookback)}D, h={int(h)}, l={int(round(float(l) * 100))}%"


def write_text(path: Path, text: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def dataframe_to_markdown(df: pd.DataFrame, float_digits: int = 6) -> str:
    display_df = df.copy()
    for col in display_df.columns:
        if pd.api.types.is_float_dtype(display_df[col]):
            display_df[col] = display_df[col].map(lambda x: "" if pd.isna(x) else f"{x:.{float_digits}f}")
        else:
            display_df[col] = display_df[col].map(lambda x: "" if pd.isna(x) else str(x))

    cols = list(display_df.columns)
    header = "| " + " | ".join(cols) + " |"
    sep = "| " + " | ".join(["---"] * len(cols)) + " |"
    rows = []
    for _, r in display_df.iterrows():
        rows.append("| " + " | ".join(str(r[c]) for c in cols) + " |")
    return "\n".join([header, sep] + rows)


def coerce_numeric(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


# ============================================================
# 2. DATA DOWNLOAD AND RETURNS
# ============================================================

def extract_close_panel(raw, requested):
    if raw is None or raw.empty:
        return pd.DataFrame()

    if isinstance(raw.columns, pd.MultiIndex):
        level0 = raw.columns.get_level_values(0)
        level1 = raw.columns.get_level_values(1)

        if "Close" in set(level0):
            close = raw["Close"].copy()
        elif "Adj Close" in set(level0):
            close = raw["Adj Close"].copy()
        elif "Close" in set(level1):
            close = raw.xs("Close", axis=1, level=1).copy()
        elif "Adj Close" in set(level1):
            close = raw.xs("Adj Close", axis=1, level=1).copy()
        else:
            raise ValueError("Could not find Close or Adj Close columns in yfinance output.")
    else:
        if "Close" in raw.columns:
            close = raw[["Close"]].copy()
            close.columns = requested[:1]
        elif "Adj Close" in raw.columns:
            close = raw[["Adj Close"]].copy()
            close.columns = requested[:1]
        else:
            raise ValueError("Could not find Close or Adj Close columns in yfinance output.")

    close.index = pd.to_datetime(close.index).tz_localize(None)
    close = close.sort_index()
    close.columns = [normalize_symbol(c) for c in close.columns]
    return close


def download_prices(tickers, start_date, end_date, batch_size=80):
    all_close = []
    symbols = [normalize_symbol(t) for t in tickers]
    symbols = list(dict.fromkeys(symbols))

    print(f"Downloading {len(symbols)} stock tickers from yfinance...")

    failed_batches = []

    for i in range(0, len(symbols), batch_size):
        batch = symbols[i:i + batch_size]
        try:
            raw = yf.download(
                tickers=batch,
                start=start_date,
                end=end_date,
                auto_adjust=True,
                progress=False,
                group_by="ticker",
                threads=True,
            )
            close = extract_close_panel(raw, batch)
            if not close.empty:
                all_close.append(close)
            print(f"  downloaded batch {i // batch_size + 1}: {len(batch)} tickers")
        except Exception as e:
            failed_batches.append({"batch_start": i, "tickers": ";".join(batch), "error": str(e)})
            print(f"  WARNING: failed batch {i // batch_size + 1}: {e}")

    if not all_close:
        raise ValueError("No stock price data downloaded from yfinance.")

    prices = pd.concat(all_close, axis=1)
    prices = prices.loc[:, ~prices.columns.duplicated()].sort_index()

    failed_df = pd.DataFrame(failed_batches)
    return prices, failed_df


def simple_returns(prices):
    return prices.sort_index().pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan)


# ============================================================
# 3. PORTFOLIO / BACKTEST HELPERS
# ============================================================

def get_eligible_stocks(train_rets):
    eligible = []
    for c in train_rets.columns:
        s = train_rets[c]
        if s.notna().all() and float(s.std(ddof=0)) > 1e-12:
            eligible.append(c)
    return eligible


def equal_weight(stocks):
    stocks = list(dict.fromkeys([s for s in stocks if isinstance(s, str)]))
    if len(stocks) == 0:
        return {}
    w = 1.0 / len(stocks)
    return {s: w for s in stocks}


def one_way_turnover(old_actual_weights, new_target_weights):
    keys = set(old_actual_weights) | set(new_target_weights)
    traded_notional = sum(
        abs(float(new_target_weights.get(k, 0.0)) - float(old_actual_weights.get(k, 0.0)))
        for k in keys
    )
    return 0.5 * traded_notional


def run_holding_period(nav_start, period_rets, target_weights, transaction_cost):
    nav_after_cost = float(nav_start) - float(transaction_cost)
    nav_after_cost = max(nav_after_cost, 0.0)

    if len(period_rets) == 0:
        return pd.Series(dtype=float), {}, nav_after_cost

    if not target_weights:
        daily_port_rets = pd.Series(0.0, index=period_rets.index)
        nav = nav_after_cost * (1.0 + daily_port_rets).cumprod()
        return nav.rename("nav"), {}, nav_after_cost

    stocks = [s for s in target_weights if s in period_rets.columns]

    if len(stocks) == 0:
        daily_port_rets = pd.Series(0.0, index=period_rets.index)
        nav = nav_after_cost * (1.0 + daily_port_rets).cumprod()
        return nav.rename("nav"), {}, nav_after_cost

    weights = pd.Series({s: target_weights[s] for s in stocks}, dtype=float)
    weights = weights / weights.sum()

    rets = period_rets[stocks].fillna(0.0)
    daily_port_rets = rets.dot(weights)

    nav = nav_after_cost * (1.0 + daily_port_rets).cumprod()
    nav = nav.rename("nav")

    gross = weights * (1.0 + rets).cumprod().iloc[-1]
    if gross.sum() > 0:
        ending_weights = (gross / gross.sum()).to_dict()
    else:
        ending_weights = weights.to_dict()

    return nav, ending_weights, nav_after_cost


def initial_nav_series(index, window_lookback, initial_nav):
    return pd.Series(
        [initial_nav],
        index=pd.DatetimeIndex([index[window_lookback - 1]]),
        name="nav",
        dtype=float,
    )


def rebalance_indices(n_rows, h, window_lookback):
    return list(range(window_lookback, n_rows, h))


def compute_drawdown(nav):
    nav = nav.dropna().astype(float)
    return (nav / nav.cummax() - 1.0).rename("drawdown")


def metrics_from_nav(nav, log_df, strategy, method, h, l, momentum_lookback=None, notes=""):
    nav = nav.dropna().astype(float)
    daily_ret = nav.pct_change(fill_method=None).dropna()

    cumulative = float(nav.iloc[-1] / nav.iloc[0] - 1.0) if len(nav) > 1 else np.nan

    years = (
        max((nav.index[-1] - nav.index[0]).days / 365.25, 1e-9)
        if len(nav) > 1
        else np.nan
    )

    annualized = (
        float((nav.iloc[-1] / nav.iloc[0]) ** (1.0 / years) - 1.0)
        if len(nav) > 1
        else np.nan
    )

    sharpe = (
        float(daily_ret.mean() / daily_ret.std(ddof=1) * np.sqrt(252))
        if len(daily_ret) > 1 and daily_ret.std(ddof=1) > 0
        else np.nan
    )

    mdd = float(compute_drawdown(nav).min()) if len(nav) > 0 else np.nan

    avg_turnover = (
        float(log_df["turnover"].mean())
        if log_df is not None and len(log_df) and "turnover" in log_df.columns
        else 0.0
    )

    avg_n = (
        float(log_df["selected_stock_count"].mean())
        if log_df is not None and len(log_df) and "selected_stock_count" in log_df.columns
        else np.nan
    )

    rb_count = int(len(log_df)) if log_df is not None else 0

    return {
        "Strategy": strategy,
        "Method": method,
        "Momentum Lookback": momentum_lookback,
        "Rebalance Frequency": h,
        "l": l,
        "Cumulative Return": cumulative,
        "Annualized Return": annualized,
        "Daily Sharpe": sharpe,
        "Daily Max Drawdown": mdd,
        "Average Turnover": avg_turnover,
        "Average Number of Stocks": avg_n,
        "Rebalance Count": rb_count,
        "Notes": notes,
    }


def series_dict_to_dataframe(series_dict):
    df = pd.concat(series_dict, axis=1)
    df.columns = [str(c) for c in df.columns]
    df = df.sort_index()
    df.index.name = "date"
    return df.reset_index()


# ============================================================
# 4. MOMENTUM SELECTION AND BACKTEST
# ============================================================

def select_momentum_stocks(train_rets, eligible_stocks, momentum_lookback, top_stock_pct):
    eligible = [s for s in eligible_stocks if s in train_rets.columns]
    if len(eligible) == 0:
        return [], pd.Series(dtype=float)

    signal_window = train_rets[eligible].tail(int(momentum_lookback)).copy()
    signal_window = signal_window.replace([np.inf, -np.inf], np.nan)

    # Require complete signal window for this specific momentum lookback.
    signal_window = signal_window.dropna(axis=1, how="any")
    if signal_window.shape[1] == 0:
        return [], pd.Series(dtype=float)

    trailing_returns = (1.0 + signal_window).prod(axis=0) - 1.0
    trailing_returns = trailing_returns.replace([np.inf, -np.inf], np.nan).dropna()

    if trailing_returns.empty:
        return [], trailing_returns

    n_select = max(1, int(math.ceil(len(trailing_returns) * float(top_stock_pct))))
    selected = trailing_returns.sort_values(ascending=False).head(n_select).index.tolist()
    return selected, trailing_returns


def run_momentum_strategy(stock_returns, h, l, momentum_lookback, window_lookback, initial_nav, transaction_cost_rate):
    nav_pieces = [initial_nav_series(stock_returns.index, window_lookback, initial_nav)]
    log_rows = []

    nav_current = float(initial_nav)
    old_actual_weights = {}

    previous_target_weights = {}
    previous_selected = []

    for rb_count, rb_idx in enumerate(rebalance_indices(len(stock_returns), h, window_lookback)):
        train_start_idx = rb_idx - window_lookback
        train_end_idx = rb_idx - 1
        hold_start_idx = rb_idx
        hold_end_idx = min(rb_idx + h - 1, len(stock_returns) - 1)

        train_rets = stock_returns.iloc[train_start_idx:rb_idx].copy()
        period_rets = stock_returns.iloc[hold_start_idx:hold_end_idx + 1].copy()

        eligible = get_eligible_stocks(train_rets)

        selected, trailing_returns = select_momentum_stocks(
            train_rets=train_rets,
            eligible_stocks=eligible,
            momentum_lookback=momentum_lookback,
            top_stock_pct=l,
        )

        fallback_used = False
        fallback_type = "none"
        fallback_reason = ""

        if len(selected) == 0:
            if previous_target_weights:
                selected = previous_selected[:]
                new_target_weights = previous_target_weights.copy()
                fallback_used = True
                fallback_type = "carry_forward_previous_portfolio"
                fallback_reason = "momentum_signal_failed_carry_forward"
            else:
                selected = []
                new_target_weights = {}
                fallback_used = True
                fallback_type = "skip_until_first_valid_signal"
                fallback_reason = "momentum_signal_failed_no_previous_portfolio"
        else:
            new_target_weights = equal_weight(selected)
            previous_target_weights = new_target_weights.copy()
            previous_selected = selected[:]

        turnover = one_way_turnover(old_actual_weights, new_target_weights)
        transaction_cost = nav_current * turnover * transaction_cost_rate
        nav_before = nav_current

        nav_period, ending_weights, nav_after_cost = run_holding_period(
            nav_start=nav_current,
            period_rets=period_rets,
            target_weights=new_target_weights,
            transaction_cost=transaction_cost,
        )

        if len(nav_period) > 0:
            nav_current = float(nav_period.iloc[-1])
            nav_pieces.append(nav_period)
        else:
            nav_current = nav_after_cost

        old_actual_weights = ending_weights

        top_signal_min = np.nan
        top_signal_max = np.nan
        top_signal_mean = np.nan
        if len(selected) > 0 and len(trailing_returns) > 0:
            selected_signals = trailing_returns.reindex(selected).dropna()
            if len(selected_signals) > 0:
                top_signal_min = float(selected_signals.min())
                top_signal_max = float(selected_signals.max())
                top_signal_mean = float(selected_signals.mean())

        log_rows.append({
            "strategy": momentum_strategy_name(momentum_lookback, h, l),
            "method": f"Momentum {int(momentum_lookback)}D",
            "momentum_lookback": int(momentum_lookback),
            "rebalance_frequency": int(h),
            "l": float(l),

            "rebalance_date": dtstr(stock_returns.index[rb_idx]),
            "train_start_date": dtstr(stock_returns.index[train_start_idx]),
            "train_end_date": dtstr(stock_returns.index[train_end_idx]),
            "hold_start_date": dtstr(stock_returns.index[hold_start_idx]),
            "hold_end_date": dtstr(stock_returns.index[hold_end_idx]),

            "n_train_days": int(len(train_rets)),
            "momentum_lookback_days": int(momentum_lookback),
            "n_eligible_stocks": int(len(eligible)),
            "n_signal_stocks": int(len(trailing_returns)),
            "selected_stock_count": int(len(selected)),

            "turnover": float(turnover),
            "transaction_cost": float(transaction_cost),
            "fallback_used": bool(fallback_used),
            "fallback_type": fallback_type,
            "fallback_reason": fallback_reason,

            "top_selected_signal_min": top_signal_min,
            "top_selected_signal_max": top_signal_max,
            "top_selected_signal_mean": top_signal_mean,

            "selected_stocks": ";".join(selected),

            "nav_before_rebalance": float(nav_before),
            "nav_after_cost": float(nav_after_cost),
            "nav_end_holding_period": float(nav_current),
        })

    nav = pd.concat(nav_pieces).sort_index()
    nav = nav[~nav.index.duplicated(keep="first")]

    return nav, pd.DataFrame(log_rows)


# ============================================================
# 5. BALANCED SCORE HELPERS FOR COMPARISON TABLE
# ============================================================

def normalized_rank_score(series: pd.Series, higher_is_better: bool) -> pd.Series:
    n = series.notna().sum()
    ranks = series.rank(
        ascending=not higher_is_better,
        method="min",
        na_option="bottom"
    )
    return (n - ranks + 1) / n


def add_balanced_score(df):
    df = df.copy()

    for c in [
        "Annualized Return",
        "Daily Sharpe",
        "Daily Max Drawdown",
        "Average Turnover",
        "Rebalance Frequency",
        "l",
    ]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df["Annualized Return Score"] = normalized_rank_score(df["Annualized Return"], higher_is_better=True)
    df["Daily Sharpe Score"] = normalized_rank_score(df["Daily Sharpe"], higher_is_better=True)
    df["Daily Max Drawdown Score"] = normalized_rank_score(df["Daily Max Drawdown"], higher_is_better=True)
    df["Average Turnover Score"] = normalized_rank_score(df["Average Turnover"], higher_is_better=False)

    df["Balanced Score"] = (
        BALANCED_SCORE_WEIGHTS["Annualized Return Score"] * df["Annualized Return Score"]
        + BALANCED_SCORE_WEIGHTS["Daily Sharpe Score"] * df["Daily Sharpe Score"]
        + BALANCED_SCORE_WEIGHTS["Daily Max Drawdown Score"] * df["Daily Max Drawdown Score"]
        + BALANCED_SCORE_WEIGHTS["Average Turnover Score"] * df["Average Turnover Score"]
    )

    df["Balanced Score Rank"] = df["Balanced Score"].rank(ascending=False, method="min").astype(int)
    return df.sort_values(["Balanced Score Rank", "Balanced Score"], ascending=[True, False]).reset_index(drop=True)


def find_representative_ranking(extract_root, exp1_extract_root=None):
    """
    Priority:
    1. step4_full_balanced_score_ranking.csv from tda_step4_exp1_outputs.zip
    2. step3_all_configurations_scored.csv from experiment_input.zip
    """
    if exp1_extract_root is not None:
        p = find_first_file(exp1_extract_root, "step4_full_balanced_score_ranking.csv")
        if p is not None:
            df = pd.read_csv(p)
            df = normalize_columns(df)
            df["source_for_comparison"] = "step4_exp1_audited_ranking"
            if "Balanced Score" not in df.columns:
                df = add_balanced_score(df)
            return df, p

    p = find_first_file(extract_root, "step3_all_configurations_scored.csv")
    if p is None:
        raise FileNotFoundError(
            "Cannot find representative-state ranking source. "
            "Expected step4_full_balanced_score_ranking.csv or step3_all_configurations_scored.csv."
        )

    df = pd.read_csv(p)
    df = normalize_columns(df)
    df = coerce_numeric(
        df,
        [
            "Rebalance Frequency",
            "l",
            "Cumulative Return",
            "Annualized Return",
            "Daily Sharpe",
            "Daily Max Drawdown",
            "Average Turnover",
            "Average Number of Stocks",
        ],
    )
    if "Strategy" not in df.columns:
        df["Strategy"] = df.apply(lambda r: strategy_name(r["Method"], r["Rebalance Frequency"], r.get("l", np.nan)), axis=1)
    df = add_balanced_score(df)
    df["source_for_comparison"] = "step3_all_configurations_scored_fallback"
    return df, p


def select_comparison_rows(representative_ranking, momentum_metrics):
    rep = representative_ranking.copy()

    for c in ["Rebalance Frequency", "l", "Balanced Score", "Balanced Score Rank"]:
        if c in rep.columns:
            rep[c] = pd.to_numeric(rep[c], errors="coerce")

    comparison_rows = []

    for method in CORE_METHODS:
        sub = rep[rep["Method"] == method].copy()
        if len(sub) == 0:
            continue
        sub = sub.sort_values(["Balanced Score Rank", "Balanced Score"], ascending=[True, False])
        row = sub.iloc[0].to_dict()
        row["Comparison Group"] = "Best representative-state configuration by audited Balanced Score"
        comparison_rows.append(row)

    for method in BENCHMARK_METHODS:
        sub = rep[rep["Method"] == method].copy()
        if len(sub) == 0:
            continue
        sub = sub.sort_values(["Balanced Score Rank", "Balanced Score"], ascending=[True, False])
        row = sub.iloc[0].to_dict()
        row["Comparison Group"] = "Best benchmark context row by audited Balanced Score"
        comparison_rows.append(row)

    mom = momentum_metrics.copy()
    mom_scored = add_balanced_score(mom)

    for lb in MOMENTUM_LOOKBACKS:
        sub = mom_scored[mom_scored["Momentum Lookback"] == lb].copy()
        if len(sub) == 0:
            continue
        sub = sub.sort_values(["Balanced Score Rank", "Balanced Score"], ascending=[True, False])
        row = sub.iloc[0].to_dict()
        row["Comparison Group"] = f"Best Momentum {lb}D configuration by within-momentum Balanced Score"
        comparison_rows.append(row)

    out = pd.DataFrame(comparison_rows)

    desired_cols = [
        "Comparison Group",
        "Strategy",
        "Method",
        "Momentum Lookback",
        "Rebalance Frequency",
        "l",
        "Cumulative Return",
        "Annualized Return",
        "Daily Sharpe",
        "Daily Max Drawdown",
        "Average Turnover",
        "Average Number of Stocks",
        "Rebalance Count",
        "Fallback Count",
        "Balanced Score",
        "Balanced Score Rank",
        "Notes",
        "source_for_comparison",
    ]

    for c in desired_cols:
        if c not in out.columns:
            out[c] = np.nan

    return out[desired_cols]


# ============================================================
# 6. MAIN EXECUTION
# ============================================================

if not INPUT_ZIP.exists():
    raise FileNotFoundError(
        f"Cannot find {INPUT_ZIP.resolve()}. "
        "Please place experiment_input.zip in the same folder as this notebook."
    )

clean_folder(WORK_ROOT)
clean_folder(OUTPUT_ROOT)

metadata_dir = OUTPUT_ROOT / "00_metadata"
momentum_dir = OUTPUT_ROOT / "02_momentum_benchmark"
metadata_dir.mkdir(parents=True, exist_ok=True)
momentum_dir.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("STEP 4 - EXPERIMENT 2: Cross-Sectional Momentum Benchmark")
print("=" * 80)

EXTRACT_ROOT = WORK_ROOT / "experiment_input_extracted"
extract_zip(INPUT_ZIP, EXTRACT_ROOT)
extract_all_nested_zips(EXTRACT_ROOT)

EXP1_EXTRACT_ROOT = None
if STEP4_EXP1_ZIP.exists():
    EXP1_EXTRACT_ROOT = WORK_ROOT / "step4_exp1_extracted"
    extract_zip(STEP4_EXP1_ZIP, EXP1_EXTRACT_ROOT)
    extract_all_nested_zips(EXP1_EXTRACT_ROOT)
    print(f"Using optional Experiment 1 zip: {STEP4_EXP1_ZIP}")
else:
    print("Optional tda_step4_exp1_outputs.zip not found. Will use Step 3 ranking fallback.")

# Copy plan/project skeleton if available
plan_path = find_first_file(EXTRACT_ROOT, "plan.md")
project_skeleton_path = find_first_file(EXTRACT_ROOT, "project_skeleton.md")

if plan_path is not None:
    shutil.copy2(plan_path, metadata_dir / "plan.md")
if project_skeleton_path is not None:
    shutil.copy2(project_skeleton_path, metadata_dir / "project_skeleton.md")

# Read Step 1 metadata
step1_metadata_path = find_first_file(EXTRACT_ROOT, "step1_experiment_metadata.json")
if step1_metadata_path is None:
    raise FileNotFoundError("Cannot find step1_experiment_metadata.json inside experiment_input.zip.")

with open(step1_metadata_path, "r", encoding="utf-8") as f:
    step1_metadata = json.load(f)

START_DATE = step1_metadata.get("start_date", "2014-01-01")
END_DATE = step1_metadata.get("end_date", "2025-12-31")
SP500_TICKER = step1_metadata.get("sp500_ticker", "^GSPC")
INITIAL_NAV = float(step1_metadata.get("initial_nav", INITIAL_NAV_DEFAULT))
WINDOW_LOOKBACK = int(step1_metadata.get("window_lookback", WINDOW_LOOKBACK_DEFAULT))
TRANSACTION_COST_RATE = float(step1_metadata.get("transaction_cost_rate", TRANSACTION_COST_RATE_DEFAULT))

valid_tickers = step1_metadata.get("valid_tickers", None)
if not valid_tickers:
    raise ValueError(
        "Step 1 metadata does not contain 'valid_tickers'. "
        "Experiment 2 requires the audited Step 1 universe to ensure consistency."
    )

valid_tickers = [normalize_symbol(t) for t in valid_tickers]
valid_tickers = list(dict.fromkeys(valid_tickers))

print(f"Universe size from Step 1 metadata: {len(valid_tickers)}")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Window lookback: {WINDOW_LOOKBACK}")
print(f"Transaction cost rate: {TRANSACTION_COST_RATE}")

# Download stock prices and compute returns
stock_prices, failed_download_batches = download_prices(valid_tickers, START_DATE, END_DATE)
stock_returns = simple_returns(stock_prices)
stock_returns = stock_returns.dropna(how="all")
stock_returns = stock_returns.loc[:, stock_returns.notna().sum() > 0]
stock_returns = stock_returns.sort_index()

if len(stock_returns) <= WINDOW_LOOKBACK + max(REBALANCE_FREQUENCIES):
    raise ValueError(
        f"Not enough return rows. Need more than {WINDOW_LOOKBACK + max(REBALANCE_FREQUENCIES)}, "
        f"got {len(stock_returns)}."
    )

print(f"Return matrix shape after download and cleaning: {stock_returns.shape}")

# Save downloaded panels for audit convenience
stock_prices.to_csv(metadata_dir / "downloaded_stock_prices_yfinance_auto_adjust_true.csv")
stock_returns.to_csv(metadata_dir / "downloaded_stock_returns_yfinance_auto_adjust_true.csv")
failed_download_batches.to_csv(metadata_dir / "failed_yfinance_batches.csv", index=False)

# Run momentum grid
nav_series = {}
metrics_rows = []
log_frames = []

for momentum_lookback in MOMENTUM_LOOKBACKS:
    for h in REBALANCE_FREQUENCIES:
        for l in SELECTION_FRACTIONS:
            label = momentum_strategy_name(momentum_lookback, h, l)
            print(f"Running {label}...")

            nav, log_df = run_momentum_strategy(
                stock_returns=stock_returns,
                h=h,
                l=l,
                momentum_lookback=momentum_lookback,
                window_lookback=WINDOW_LOOKBACK,
                initial_nav=INITIAL_NAV,
                transaction_cost_rate=TRANSACTION_COST_RATE,
            )

            nav_series[label] = nav
            log_frames.append(log_df)

            metrics_rows.append(
                metrics_from_nav(
                    nav=nav,
                    log_df=log_df,
                    strategy=label,
                    method=f"Momentum {momentum_lookback}D",
                    h=h,
                    l=l,
                    momentum_lookback=momentum_lookback,
                    notes=(
                        "Cross-sectional momentum benchmark. "
                        "At each rebalance, stocks are ranked by trailing cumulative return ending at tau-1. "
                        "Eligible stocks require complete data in the 504-day lookback window. "
                        "Average Turnover is average one-way rebalancing turnover, not average daily turnover."
                    ),
                )
            )

momentum_metrics = pd.DataFrame(metrics_rows)
momentum_log = pd.concat(log_frames, ignore_index=True) if log_frames else pd.DataFrame()

# Build daily NAV and returns
daily_nav = series_dict_to_dataframe(nav_series)

daily_nav_indexed = daily_nav.set_index("date").sort_index()
daily_nav_indexed = daily_nav_indexed.ffill()

daily_returns_indexed = daily_nav_indexed.pct_change(fill_method=None).fillna(0.0)

daily_nav = daily_nav_indexed.reset_index()
daily_returns = daily_returns_indexed.reset_index()

# Comparison table against representative-state methods
representative_ranking, representative_ranking_source = find_representative_ranking(
    EXTRACT_ROOT,
    EXP1_EXTRACT_ROOT,
)

comparison_table = select_comparison_rows(representative_ranking, momentum_metrics)

# Save required outputs
metrics_path = momentum_dir / "step4_momentum_benchmark_metrics.csv"
daily_nav_path = momentum_dir / "step4_momentum_benchmark_daily_nav.csv"
daily_returns_path = momentum_dir / "step4_momentum_benchmark_daily_returns.csv"
log_path = momentum_dir / "step4_momentum_benchmark_rebalance_log.csv"
comparison_md_path = momentum_dir / "step4_momentum_vs_representative_state_table.md"

momentum_metrics.to_csv(metrics_path, index=False)
daily_nav.to_csv(daily_nav_path, index=False)
daily_returns.to_csv(daily_returns_path, index=False)
momentum_log.to_csv(log_path, index=False)

comparison_md = "# Step 4 Experiment 2 — Momentum vs Representative-State Comparison\n\n"
comparison_md += "## Design\n\n"
comparison_md += (
    "At each rebalance date, the momentum benchmark ranks eligible stocks by trailing cumulative return "
    "ending at the day before rebalance, selects the top l fraction, equal-weights selected stocks, "
    "holds until the next rebalance, and applies the same 10 bps one-way turnover cost convention used "
    "in the representative-state experiments.\n\n"
)
comparison_md += "## Required Comparison Table\n\n"
comparison_md += dataframe_to_markdown(
    comparison_table[
        [
            "Comparison Group",
            "Strategy",
            "Method",
            "Momentum Lookback",
            "Rebalance Frequency",
            "l",
            "Cumulative Return",
            "Annualized Return",
            "Daily Sharpe",
            "Daily Max Drawdown",
            "Average Turnover",
            "Average Number of Stocks",
            "Balanced Score",
            "Balanced Score Rank",
        ]
    ],
    float_digits=6
)
comparison_md += "\n\n"
comparison_md += "## Notes\n\n"
comparison_md += f"- Representative-state ranking source: `{representative_ranking_source}`\n"
comparison_md += "- Momentum best rows are selected by Balanced Score within the 27 momentum configurations generated here.\n"
comparison_md += "- This table is descriptive; it is not a statistical significance test.\n"

write_text(comparison_md_path, comparison_md)

# Copy required files to output root for easy upload/audit
for p in [metrics_path, daily_nav_path, daily_returns_path, log_path, comparison_md_path]:
    shutil.copy2(p, OUTPUT_ROOT / p.name)

# Additional useful audit files
comparison_table.to_csv(momentum_dir / "step4_momentum_vs_representative_state_table.csv", index=False)
comparison_table.to_csv(OUTPUT_ROOT / "step4_momentum_vs_representative_state_table.csv", index=False)

run_config = {
    "step": "Step 4",
    "experiment": "Experiment 2 - Cross-Sectional Momentum Benchmark",
    "input_zip": str(INPUT_ZIP),
    "input_zip_sha256": sha256_file(INPUT_ZIP),
    "step4_exp1_zip_present": STEP4_EXP1_ZIP.exists(),
    "step4_exp1_zip_sha256": sha256_file(STEP4_EXP1_ZIP) if STEP4_EXP1_ZIP.exists() else "",
    "representative_ranking_source": str(representative_ranking_source),
    "start_date": START_DATE,
    "end_date": END_DATE,
    "window_lookback": WINDOW_LOOKBACK,
    "initial_nav": INITIAL_NAV,
    "transaction_cost_rate": TRANSACTION_COST_RATE,
    "rebalance_frequencies": REBALANCE_FREQUENCIES,
    "selection_fractions": SELECTION_FRACTIONS,
    "momentum_lookbacks": MOMENTUM_LOOKBACKS,
    "stock_universe_size_from_step1_metadata": len(valid_tickers),
    "downloaded_price_columns": int(stock_prices.shape[1]),
    "return_matrix_shape": list(stock_returns.shape),
    "price_source": "yfinance yf.download(auto_adjust=True)",
    "generated_at": datetime.now().isoformat(timespec="seconds"),
}

with open(metadata_dir / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2, ensure_ascii=False)

# Artifact check
artifact_rows = []
for fname in REQUIRED_OUTPUTS:
    p1 = OUTPUT_ROOT / fname
    p2 = momentum_dir / fname
    p = p1 if p1.exists() else p2
    artifact_rows.append({
        "file_name": fname,
        "exists": p.exists(),
        "size_bytes": p.stat().st_size if p.exists() else 0,
        "path": str(p),
        "sha256": sha256_file(p) if p.exists() else "",
    })

artifact_check = pd.DataFrame(artifact_rows)
artifact_check.to_csv(OUTPUT_ROOT / "step4_exp2_artifact_file_check.csv", index=False)

audit_text = "# Step 4 Experiment 2 — Momentum Benchmark Audit Summary\n\n"
audit_text += f"- Generated at: `{datetime.now().isoformat(timespec='seconds')}`\n"
audit_text += f"- Input zip SHA256: `{sha256_file(INPUT_ZIP)}`\n"
audit_text += f"- Representative ranking source: `{representative_ranking_source}`\n"
audit_text += f"- Universe size from Step 1 metadata: `{len(valid_tickers)}`\n"
audit_text += f"- Downloaded price columns: `{stock_prices.shape[1]}`\n"
audit_text += f"- Return matrix shape: `{stock_returns.shape}`\n"
audit_text += f"- Momentum configurations expected: `27`\n"
audit_text += f"- Momentum configurations generated: `{len(momentum_metrics)}`\n"
audit_text += f"- Rebalance log rows: `{len(momentum_log)}`\n"
audit_text += f"- Failed yfinance batches: `{len(failed_download_batches)}`\n\n"

audit_text += "## Required Artifact Check\n\n"
audit_text += dataframe_to_markdown(artifact_check, float_digits=0)
audit_text += "\n\n"

audit_text += "## Best Momentum Configuration by Balanced Score\n\n"
mom_scored_for_audit = add_balanced_score(momentum_metrics)
best_mom = mom_scored_for_audit.sort_values(["Balanced Score Rank", "Balanced Score"], ascending=[True, False]).head(10)
audit_text += dataframe_to_markdown(
    best_mom[
        [
            "Balanced Score Rank",
            "Strategy",
            "Method",
            "Momentum Lookback",
            "Rebalance Frequency",
            "l",
            "Cumulative Return",
            "Annualized Return",
            "Daily Sharpe",
            "Daily Max Drawdown",
            "Average Turnover",
            "Average Number of Stocks",
            "Balanced Score",
        ]
    ],
    float_digits=6
)
audit_text += "\n\n"

audit_text += "## Interpretation Guardrail\n\n"
audit_text += (
    "This experiment only provides a direct momentum benchmark. "
    "Do not claim representative-state selection adds information beyond momentum until the comparison table is audited. "
    "Do not make statistical superiority claims until the bootstrap experiment is completed.\n"
)

write_text(OUTPUT_ROOT / "step4_momentum_benchmark_audit.md", audit_text)
write_text(momentum_dir / "step4_momentum_benchmark_audit.md", audit_text)

# Zip output
zip_output = Path("tda_step4_exp2_outputs.zip")
if zip_output.exists():
    zip_output.unlink()

shutil.make_archive(
    base_name=str(zip_output.with_suffix("")),
    format="zip",
    root_dir=OUTPUT_ROOT
)

print("=" * 80)
print("DONE - Step 4 Experiment 2 outputs generated.")
print(f"Output folder: {OUTPUT_ROOT.resolve()}")
print(f"Output zip: {zip_output.resolve()}")
print("=" * 80)

print("\nArtifact check:")
display(artifact_check)

print("\nMomentum metrics preview:")
display(
    momentum_metrics[
        [
            "Strategy",
            "Method",
            "Momentum Lookback",
            "Rebalance Frequency",
            "l",
            "Cumulative Return",
            "Annualized Return",
            "Daily Sharpe",
            "Daily Max Drawdown",
            "Average Turnover",
            "Average Number of Stocks",
        ]
    ].head(10)
)

print("\nComparison table preview:")
display(
    comparison_table[
        [
            "Comparison Group",
            "Strategy",
            "Method",
            "Momentum Lookback",
            "Rebalance Frequency",
            "l",
            "Annualized Return",
            "Daily Sharpe",
            "Daily Max Drawdown",
            "Average Turnover",
            "Balanced Score",
            "Balanced Score Rank",
        ]
    ]
)

STEP 4 - EXPERIMENT 2: Cross-Sectional Momentum Benchmark
Using optional Experiment 1 zip: tda_step4_exp1_outputs.zip
Universe size from Step 1 metadata: 359
Date range: 2014-01-01 to 2025-12-31
Window lookback: 504
Transaction cost rate: 0.001
  downloaded batch 1: 80 tickers
  downloaded batch 2: 80 tickers
  downloaded batch 3: 80 tickers
  downloaded batch 4: 80 tickers
  downloaded batch 5: 39 tickers
Return matrix shape after download and cleaning: (3016, 359)
Running Momentum 21D, h=21, l=20%...
Running Momentum 21D, h=21, l=30%...
Running Momentum 21D, h=21, l=40%...
Running Momentum 21D, h=42, l=20%...
Running Momentum 21D, h=42, l=30%...
Running Momentum 21D, h=42, l=40%...
Running Momentum 21D, h=63, l=20%...
Running Momentum 21D, h=63, l=30%...
Running Momentum 21D, h=63, l=40%...
Running Momentum 63D, h=21, l=20%...
Running Momentum 63D, h=21, l=30%...
Running Momentum 63D, h=21, l=40%...
Running Momentum 63D, h=42, l=20%...
Running Momentum 63D, h=42, l=30%...
Running Mom

,file_name,exists,size_bytes,path,sha256
0,step4_momentum_benchmark_metrics.csv,True,12188,tda_step4_exp2_outputs\step4_momentum_benchmar...,9a2f2632b61d4b1f0929e0563fc67df76d0fa316903a3d...
1,step4_momentum_benchmark_daily_nav.csv,True,1292211,tda_step4_exp2_outputs\step4_momentum_benchmar...,b9366f52055fc68be52e783173635e3b117544a9f2f242...
2,step4_momentum_benchmark_daily_returns.csv,True,1499386,tda_step4_exp2_outputs\step4_momentum_benchmar...,5aafea3788df41087b4f198d7cc98d7d269717e63ef182...
3,step4_momentum_benchmark_rebalance_log.csv,True,1450550,tda_step4_exp2_outputs\step4_momentum_benchmar...,e95236aa73ed2f32e9e16d3b1135d105c597d35d164e81...
4,step4_momentum_vs_representative_state_table.md,True,2799,tda_step4_exp2_outputs\step4_momentum_vs_repre...,88f2814c51893dbab6e7fae0712bea3e025d1b48db9189...



Momentum metrics preview:


,Strategy,Method,Momentum Lookback,Rebalance Frequency,l,Cumulative Return,Annualized Return,Daily Sharpe,Daily Max Drawdown,Average Turnover,Average Number of Stocks
0,"Momentum 21D, h=21, l=20%",Momentum 21D,21,21,0.2,1.842819,0.110276,0.653040,-0.322188,0.800702,70.850000
1,"Momentum 21D, h=21, l=30%",Momentum 21D,21,21,0.3,1.797757,0.108501,0.660849,-0.326793,0.711157,106.158333
2,"Momentum 21D, h=21, l=40%",Momentum 21D,21,21,0.4,2.085174,0.119408,0.727178,-0.326279,0.609332,141.058333
3,"Momentum 21D, h=42, l=20%",Momentum 21D,21,42,0.2,2.526540,0.134495,0.780491,-0.370118,0.782445,70.833333
4,"Momentum 21D, h=42, l=30%",Momentum 21D,21,42,0.3,2.514927,0.134120,0.796726,-0.367117,0.696987,106.133333
5,"Momentum 21D, h=42, l=40%",Momentum 21D,21,42,0.4,2.789997,0.142708,0.851695,-0.360646,0.603016,141.033333
6,"Momentum 21D, h=63, l=20%",Momentum 21D,21,63,0.2,1.625651,0.101477,0.589753,-0.399785,0.771287,70.825000
7,"Momentum 21D, h=63, l=30%",Momentum 21D,21,63,0.3,1.818401,0.109317,0.645262,-0.392446,0.684140,106.150000
8,"Momentum 21D, h=63, l=40%",Momentum 21D,21,63,0.4,2.000711,0.116301,0.687848,-0.391548,0.605304,141.000000
9,"Momentum 63D, h=21, l=20%",Momentum 63D,63,21,0.2,3.050172,0.150330,0.858036,-0.331358,0.471114,70.850000



Comparison table preview:


,Comparison Group,Strategy,Method,Momentum Lookback,Rebalance Frequency,l,Annualized Return,Daily Sharpe,Daily Max Drawdown,Average Turnover,Balanced Score,Balanced Score Rank
0,Best representative-state configuration by aud...,"Mapper, h=21, l=20%",Mapper,NaN,21,0.2,0.175150,0.970123,-0.358387,0.167930,0.851515,3
1,Best representative-state configuration by aud...,"Global DBSCAN, h=21, l=20%",Global DBSCAN,NaN,21,0.2,0.169831,0.944330,-0.366202,0.151404,0.660606,11
2,Best representative-state configuration by aud...,"PCA-KMeans, h=21, l=20%",PCA-KMeans,NaN,21,0.2,0.179900,0.998143,-0.355070,0.172253,0.909091,1
3,Best benchmark context row by audited Balanced...,"Equal Weight Universe, h=63",Equal Weight Universe,NaN,63,NaN,0.162121,0.916253,-0.373879,0.057573,0.259091,30
4,Best benchmark context row by audited Balanced...,"S&P 500, h=21",S&P 500,NaN,21,NaN,0.131228,0.772522,-0.339250,0.000000,0.363636,23
5,Best Momentum 21D configuration by within-mome...,"Momentum 21D, h=42, l=40%",Momentum 21D,21.0,42,0.4,0.142708,0.851695,-0.360646,0.603016,0.637037,13
6,Best Momentum 63D configuration by within-mome...,"Momentum 63D, h=21, l=20%",Momentum 63D,63.0,21,0.2,0.150330,0.858036,-0.331358,0.471114,0.703704,8
7,Best Momentum 126D configuration by within-mom...,"Momentum 126D, h=42, l=20%",Momentum 126D,126.0,42,0.2,0.172713,0.924100,-0.364801,0.469903,0.881481,1
